In [1]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table as table, join
from astropy.io import fits
from astropy.wcs import WCS
import matplotlib as mpl
import os
import scipy.stats
import glob
%matplotlib inline
from astropy.nddata import CCDData
import astropy.units as u
import urllib.request

In [2]:
def imdisplay(image, v1perc=10, v2perc=95):
    '''
    display an image
    OPTIONAL KEYWORD PARAMETERS
    v1perc: one end of the colormap assigned to the v1perc percent lowest flux
    v2perc: the other end of the colormap assigned to the v2perc percent highest flux
    '''
    # make sure image is an np array
    nimage = np.array(image)
    # determine the pixel values at the 10th and 95th percentile
    v1 = scoreatpercentile(nimage,v1perc)
    v2 = scoreatpercentile(nimage,v2perc)
    # display using imshow
    #
    # you can play with alternate cmaps in the function below, such as "viridis" or "gray"
    # The 'gray_r' color map reverses the color-scale so that dark display pixels are the brightest in the image
    #
    # vmin and vmax set the min and max pixel values that
    # will be mapped to the extremes of the colormap
    plt.imshow(nimage,cmap='grey_r',vmin=v1,vmax=v2)
    plt.colorbar(fraction=.08)

In [3]:
# Getting image files
base_url = "https://portal.nersc.gov/cfs/cosmo/data/legacysurvey/dr11/south/coadd"

files_M67 = {
    "M67_topleft_g": f"{base_url}/132/1328p120/legacysurvey-1328p120-image-g.fits.fz",
    "M67_topleft_r": f"{base_url}/132/1328p120/legacysurvey-1328p120-image-r.fits.fz",

    "M67_topright_g": f"{base_url}/132/1326p120/legacysurvey-1326p120-image-g.fits.fz",
    "M67_topright_r": f"{base_url}/132/1326p120/legacysurvey-1326p120-image-r.fits.fz",

    "M67_bottomleft_g": f"{base_url}/132/1329p117/legacysurvey-1329p117-image-g.fits.fz",
    "M67_bottomleft_r": f"{base_url}/132/1329p117/legacysurvey-1329p117-image-r.fits.fz",

    "M67_bottomright_g": f"{base_url}/132/1327p117/legacysurvey-1327p117-image-g.fits.fz",
    "M67_bottomright_r": f"{base_url}/132/1327p117/legacysurvey-1327p117-image-r.fits.fz",
}

# files_M67_trim = {
#     "M67_topleft_g": fits.getdata(files_M67['M67_topleft_g'], header=True)[1800:][:1800],
#     "M67_topleft_r": fits.getdata(files_M67['M67_topleft_r'], header=True)[1800:][:1800],

#     "M67_topright_g": fits.getdata(files_M67['M67_topright_g'], header=True)[:1800][:1800],
#     "M67_topright_r": fits.getdata(files_M67['M67_topright_r'], header=True)[:1800][:1800],

#     "M67_bottomleft_g": fits.getdata(files_M67['M67_bottomleft_g'], header=True)[1800:][1800:],
#     "M67_bottomleft_r": fits.getdata(files_M67['M67_bottomleft_r'], header=True)[1800:][1800:],

#     "M67_bottomright_g": fits.getdata(files_M67['M67_bottomright_g'], header=True)[:1800][1800:],
#     "M67_bottomright_r": fits.getdata(files_M67['M67_bottomright_r'], header=True)[:1800][1800:],
# }

files_M3 = {
    "M3_topleft_g": f"{base_url}/205/2056p285/legacysurvey-2056p285-image-g.fits.fz",
    "M3_topleft_r": f"{base_url}/205/2056p285/legacysurvey-2056p285-image-r.fits.fz",

    "M3_topright_g": f"{base_url}/205/2054p285/legacysurvey-2054p285-image-g.fits.fz",
    "M3_topright_r": f"{base_url}/205/2054p285/legacysurvey-2054p285-image-r.fits.fz",

    "M3_bottomleft_g": f"{base_url}/205/2056p282/legacysurvey-2056p282-image-g.fits.fz",
    "M3_bottomleft_r": f"{base_url}/205/2056p282/legacysurvey-2056p282-image-r.fits.fz",

    "M3_bottomright_g": f"{base_url}/205/2053p282/legacysurvey-2053p282-image-g.fits.fz",
    "M3_bottomright_r": f"{base_url}/205/2053p282/legacysurvey-2053p282-image-r.fits.fz",
}

# files_M3_trim = {
#     "M3_topleft_g": fits.getdata(files_M67['M67_topleft_g'], header=True)[1800:][:1800],
#     "M3_topleft_r": fits.getdata(files_M67['M67_topleft_r'], header=True)[1800:][:1800],

#     "M3_topright_g": fits.getdata(files_M67['M67_topright_g'], header=True)[:1800][:1800],
#     "M3_topright_r": fits.getdata(files_M67['M67_topright_r'], header=True)[:1800][:1800],

#     "M3_bottomleft_g": fits.getdata(files_M67['M67_bottomleft_g'], header=True)[1800:][1800:],
#     "M3_bottomleft_r": fits.getdata(files_M67['M67_bottomleft_r'], header=True)[1800:][1800:],

#     "M3_bottomright_g": fits.getdata(files_M67['M67_bottomright_g'], header=True)[:1800][1800:],
#     "M3_bottomright_r": fits.getdata(files_M67['M67_bottomright_r'], header=True)[:1800][1800:],
# }


In [4]:
# Downloading images
os.makedirs("/home/s310m868/ASTR596/My_respositories/Capstone/fits_data", exist_ok=True)

local_files = {}

for name, url in files_M67.items():
    path = f"/home/s310m868/ASTR596/My_respositories/Capstone/fits_data/{name}.fits.fz"

    if not os.path.exists(path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path)
    # print(f"Downloading {name}...")
    # urllib.request.urlretrieve(url, path)

    local_files[name] = path

for name, url in files_M3.items():
    path = f"/home/s310m868/ASTR596/My_respositories/Capstone/fits_data/{name}.fits.fz"

    if not os.path.exists(path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, path)
    # print(f"Downloading {name}...")
    # urllib.request.urlretrieve(url, path)
    
    local_files[name] = path

print("All files downloaded.")

All files downloaded.


In [5]:
# Get structure and header information
image_data = {}
image_headers = {}

keywords = [
    "BUNIT", "FILTER", "NAXIS1", "NAXIS2",
    "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2",
    "CD1_1", "CD1_2", "CD2_1", "CD2_2"
]

for name, path in local_files.items():
    with fits.open(path) as hdul:
        print(f"\n{'='*60}")
        print(name)
        hdul.info()

        # Find the first 2D image HDU
        image_hdu = next(
            hdu for hdu in hdul
            if hdu.data is not None and hdu.data.ndim == 2
        )

        image_data[name] = image_hdu.data
        image_headers[name] = image_hdu.header

        print("Image shape:", image_hdu.data.shape)

        for key in keywords:
            print(f"{key}: {image_hdu.header.get(key, 'Not available')}")


M67_topleft_g
Filename: /home/s310m868/ASTR596/My_respositories/Capstone/fits_data/M67_topleft_g.fits.fz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  IMAGE_g       1 CompImageHDU    133   (3600, 3600)   float32   
Image shape: (3600, 3600)
BUNIT: nanomaggy
FILTER: g DECam SDSS c0001 4720.0 1520.0
NAXIS1: 3600
NAXIS2: 3600
CRVAL1: 132.893617021277
CRVAL2: 12.0
CRPIX1: 1800.5
CRPIX2: 1800.5
CD1_1: -7.27777777777778e-05
CD1_2: 0.0
CD2_1: 0.0
CD2_2: 7.27777777777778e-05

M67_topleft_r
Filename: /home/s310m868/ASTR596/My_respositories/Capstone/fits_data/M67_topleft_r.fits.fz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  IMAGE_r       1 CompImageHDU    133   (3600, 3600)   float32   
Image shape: (3600, 3600)
BUNIT: nanomaggy
FILTER: r DECam SDSS c0002 6415.0 1480.0
NAXIS1: 3600
NAXIS2: 3600
CRVAL1: 132.893617021277
CRVAL2: 12.0
CRPIX1: 1800.5
CRPI

In [6]:
# # Display images
# fig, axes = plt.subplots(8, 2, figsize=(14, 40))

# for ax, (name, data) in zip(axes.flat, image_data.items()):
#     vmin = np.nanpercentile(data, 5)
#     vmax = np.nanpercentile(data, 99.5)

#     ax.imshow(
#         data,
#         origin="lower",
#         cmap="gray",
#         vmin=vmin,
#         vmax=vmax
#     )

#     ax.set_title(name)
#     ax.set_xlabel("X pixel")
#     ax.set_ylabel("Y pixel")

# plt.tight_layout()
# plt.show()

In [7]:
from photutils.detection import DAOStarFinder
from astropy.stats import sigma_clipped_stats
from photutils.aperture import CircularAperture
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.io import fits
import glob
import os
from scipy.stats import scoreatpercentile

from matplotlib import pyplot as plt
import numpy as np

#import ccdproc
from astropy.nddata import CCDData
import astropy.units as u
from ccdproc import ImageFileCollection, Combiner, combine
from ccdproc import wcs_project
from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd
from reproject import reproject_interp

In [8]:
# start of the combination process -- from Reductionproject

m3_rimage_list = glob.glob('*/M3*r.fits.fz')
m3_gimage_list = glob.glob('*/M3*g.fits.fz')
m67_rimage_list = glob.glob('*/M67*r.fits.fz')
m67_gimage_list = glob.glob('*/M67*g.fits.fz')

m3_rimage, m3_rheader = fits.getdata(m3_rimage_list[0],header=True)
m67_rimage, m67_rheader = fits.getdata(m67_rimage_list[0],header=True)

m3_target_wcs = WCS(m3_rheader)
m67_target_wcs = WCS(m67_rheader)


print(m3_target_wcs)
print(m67_target_wcs)

print(m3_rimage_list)
print(m3_gimage_list)
print(m67_rimage_list)
print(m67_gimage_list)


WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN'
CUNIT : 'deg' 'deg'
CRVAL : 205.410094637224 28.5
CRPIX : 1800.5 1800.5
CD1_1 CD1_2  : -7.27777777777778e-05 0.0
CD2_1 CD2_2  : 0.0 7.27777777777778e-05
NAXIS : 3600  3600
WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN'
CUNIT : 'deg' 'deg'
CRVAL : 132.63829787234 12.0
CRPIX : 1800.5 1800.5
CD1_1 CD1_2  : -7.27777777777778e-05 0.0
CD2_1 CD2_2  : 0.0 7.27777777777778e-05
NAXIS : 3600  3600
['fits_data/M3_topright_r.fits.fz', 'fits_data/M3_bottomleft_r.fits.fz', 'fits_data/M3_bottomright_r.fits.fz', 'fits_data/M3_topleft_r.fits.fz']
['fits_data/M3_topleft_g.fits.fz', 'fits_data/M3_topright_g.fits.fz', 'fits_data/M3_bottomright_g.fits.fz', 'fits_data/M3_bottomleft_g.fits.fz']
['fits_data/M67_topright_r.fits.fz', 'fits_data/M67_topleft_r.fits.fz', 'fits_data/M67_bottomleft_r.fits.fz', 'fits_data/M67_bottomright_r.fits.fz']
['fits_data/M67_topright_g.fits.fz', 'fits_data/M67_bottomright_g.fits.fz', 'fits_data

In [9]:
with fits.open("fits_data/M3_topright_r.fits.fz") as hdul:
  print(hdul.info())

CCDData.read('fits_data/M3_topright_r.fits.fz', hdu=1)
    
# CCDData.read('fits_data/M3_topright_r.fits.fz')

Filename: fits_data/M3_topright_r.fits.fz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  IMAGE_r       1 CompImageHDU    133   (3600, 3600)   float32   
None


CCDData([[ 4.9467921e+00,  6.0110831e+00,  6.6663270e+00, ...,
          -1.4080305e-03, -7.0055695e-03, -9.5473630e-03],
         [ 4.6720400e+00,  5.5511270e+00,  6.0302496e+00, ...,
           3.0143606e-03, -4.6753245e-03,  4.4808704e-03],
         [ 4.2465792e+00,  4.8766761e+00,  5.1456556e+00, ...,
           4.6459230e-05, -6.5958416e-03,  7.3499805e-03],
         ...,
         [-1.6579265e-04, -1.0634582e-02,  1.4193282e-03, ...,
           6.1758128e-03,  3.2199621e-03, -8.4646735e-03],
         [ 5.9234132e-03,  6.5381447e-04, -3.7973740e-03, ...,
          -1.6382808e-05,  8.4401254e-05, -1.0037997e-02],
         [-8.7328558e-04,  3.5315792e-03,  8.0952160e-03, ...,
          -4.5971484e-03, -8.4470809e-03,  8.3442274e-03]],
        shape=(3600, 3600), dtype=float32, unit='nmgy')

In [ ]:
# We will reproject all the images to align them with the first image in each of the lists for a given filter.
#start with the R-band data
reprojected = []
projected_ccds = []
#loop through each image name
for image_name in m3_rimage_list[0:5]:
    #read the image into a "CCDData" object
    print(image_name)
    img = CCDData.read(image_name, hdu=1)
    #take each image, and reproject it onto an identical coordinate grid to the target image
    # new_image = wcs_project(img, m3_target_wcs)
    #this is a list of all the reprojected images
    # reprojected.append(new_image)
    projected_ccds.append(img)
    # print(img)
print('done1')
print(projected_ccds)


# m3_target_wcs, m3_target_shape = find_optimal_celestial_wcs(projected_ccds, auto_rotate=True)

m3_r_hdus = [fits.open(f) for f in m3_rimage_list[0:5]]
print(m3_r_hdus)
print(type(m3_r_hdus))
m3_target_wcs, m3_target_shape = find_optimal_celestial_wcs(m3_r_hdus, hdu_in=1)


reprojected_ccds = [wcs_project(ccd, m3_target_wcs, target_shape=m3_target_shape) for ccd in projected_ccds]


# m3_rstacked_image = combine(reprojected_ccds, method='average')


m3_rstacked_image, footprint = reproject_and_coadd(m3_r_hdus, m3_target_wcs, shape_out=m3_target_shape, reproject_function=reproject_interp, match_background=True, hdu_in=1)
    


# # combine the reprojected images into one image
# # combiner = Combiner(reprojected)
# print('done2.1')
# # m3_rstacked_image = combiner.average_combine()
# m3_rstacked_image = combine(reprojected, method='average')
# print('done2.2')

# # save the combined image
# # fits.writeto('/Users/terrymechem/jupyter_notebooks/ASTR591_code/capstone/test.fits',m3_rstacked_image,overwrite=True,header=m3_rheader)
fits.writeto('fits_data/combtest5.fits',m3_rstacked_image,overwrite=True,header=m3_rheader)
# print('done3')


# display the combined image
plt.figure()
imdisplay(m3_rstacked_image)
plt.title('M3 R-band image')

fits_data/M3_topright_r.fits.fz
fits_data/M3_bottomleft_r.fits.fz
fits_data/M3_bottomright_r.fits.fz
fits_data/M3_topleft_r.fits.fz
done1
[CCDData([[ 4.9467921e+00,  6.0110831e+00,  6.6663270e+00, ...,
          -1.4080305e-03, -7.0055695e-03, -9.5473630e-03],
         [ 4.6720400e+00,  5.5511270e+00,  6.0302496e+00, ...,
           3.0143606e-03, -4.6753245e-03,  4.4808704e-03],
         [ 4.2465792e+00,  4.8766761e+00,  5.1456556e+00, ...,
           4.6459230e-05, -6.5958416e-03,  7.3499805e-03],
         ...,
         [-1.6579265e-04, -1.0634582e-02,  1.4193282e-03, ...,
           6.1758128e-03,  3.2199621e-03, -8.4646735e-03],
         [ 5.9234132e-03,  6.5381447e-04, -3.7973740e-03, ...,
          -1.6382808e-05,  8.4401254e-05, -1.0037997e-02],
         [-8.7328558e-04,  3.5315792e-03,  8.0952160e-03, ...,
          -4.5971484e-03, -8.4470809e-03,  8.3442274e-03]],
        shape=(3600, 3600), dtype=float32, unit='nmgy'), CCDData([[-0.00581043, -0.00046353, -0.00373397, ...,  0.